# Phase 1.2: Data Cleaning & Preprocessing

## Objectif
Nettoyer, interpoler, lisser et ingénierer les features sur le dataset fusionné brut.

## Sortie
- `ia/data/cleaned_features.csv`: Dataset nettoyé avec features
- `ia/models/scaler.pkl`: MinMaxScaler global pour normalisation

## Section 1: Setup et Chargement

In [1]:
# ========================================
# SECTION 1: Imports et Configuration
# ========================================
# Bibliothèques pour nettoyage et preprocessing des données

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats
import joblib
from pathlib import Path

# Fixed random seed pour reproductibilité
np.random.seed(42)

# Style des graphiques
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ Imports réussis")
print(f"   scikit-learn version: 1.6+")
print(f"   Scalers disponibles: MinMaxScaler, StandardScaler")

✅ Imports réussis
   scikit-learn version: 1.6+
   Scalers disponibles: MinMaxScaler, StandardScaler


In [2]:
# ========================================
# SECTION 2: Configuration des chemins
# ========================================
# Lier les fichiers INPUT et OUTPUT pour ce notebook

PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# Créer répertoire models s'il n'existe pas
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ENTRÉE: Fichier brut fusionné du notebook 01
raw_file = DATA_DIR / "raw_merged.csv"

# SORTIES: Fichiers traités et objets sauvegardes
cleaned_file = DATA_DIR / "cleaned_features.csv"
output_file = cleaned_file  # Pour compatibilité avec le reste du code
scaler_file = MODELS_DIR / "scaler.pkl"

print(f"📂 Input: {raw_file}")
print(f"📂 Output cleaned: {cleaned_file}")
print(f"📂 Output scaler: {scaler_file}")

📂 Input: C:\Users\melik\Desktop\ia\data\raw_merged.csv
📂 Output cleaned: C:\Users\melik\Desktop\ia\data\cleaned_features.csv
📂 Output scaler: C:\Users\melik\Desktop\ia\models\scaler.pkl


In [3]:
# ========================================
# SECTION 3: Chargement du dataset brut
# ========================================
# Charger le fichier fusionné du notebook 01

print("🔄 Chargement données brutes...\n")

TARGET_POLLUTANTS = ['NOX', 'SOX', 'PM25', 'PM10', 'CO2', 'COV']
EXCLUDED_POLLUTANTS = {'CO', 'O3', 'AQI', 'NO2', 'NOx', 'SO2', 'PM2.5'}

POLLUTANT_ALIASES = {
    'NO2': 'NOX', 'NOX': 'NOX', 'NOx': 'NOX',
    'SO2': 'SOX', 'SOX': 'SOX',
    'PM2.5': 'PM25', 'PM25': 'PM25',
    'PM10': 'PM10', 'CO2': 'CO2',
    'COV': 'COV', 'VOC': 'COV', 'VOCS': 'COV',
}

def normalize_pollutant_name(name):
    if pd.isna(name):
        return None
    key = str(name).strip().upper().replace('.', '')
    if key in EXCLUDED_POLLUTANTS and key not in POLLUTANT_ALIASES:
        return None
    return POLLUTANT_ALIASES.get(key, key if key in TARGET_POLLUTANTS else None)

df = pd.read_csv(raw_file)
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], errors='coerce')
df['pollutant'] = df['pollutant'].apply(normalize_pollutant_name)
before = len(df)
df = df[df['pollutant'].isin(TARGET_POLLUTANTS)].dropna(subset=['timestamp_utc', 'value'])
df_raw = df.copy()

print(f"✅ Dataset chargé: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"   Lignes exclues (CO/O3/autres): {before - len(df):,}")
print(f"   Polluants: {sorted(df['pollutant'].unique().tolist())}")
print(f"   Sites: {df['site_id'].nunique()} sites")
print(f"   Manquantes (NaN): {df['value'].isna().sum():,} valeurs")

🔄 Chargement données brutes...



✅ Dataset chargé: 134,756 lignes × 14 colonnes
   Lignes exclues (CO/O3/autres): 0
   Polluants: ['COV', 'NOX', 'PM10', 'PM25', 'SOX']
   Sites: 6 sites
   Manquantes (NaN): 0 valeurs


## Section 2: Nettoyage et Interpolation

In [4]:
# Interpolation linéaire pour trous courts (<5% par séquence)
print("🔄 Interpolation des valeurs manquantes...\n")

df = df_raw.copy()

# Interpoler par site et polluant
df_interpolated = []

for (site, pollutant), group in df.groupby(['site_id', 'pollutant']):
    group = group.sort_values('timestamp_utc').copy()
    
    # Interpolation linéaire
    group['value'] = group['value'].interpolate(method='linear', limit=10)
    
    # Marquer les valeurs interpolées
    group['imputed'] = group['value'].isna().shift(1).fillna(False)
    
    df_interpolated.append(group)

df = pd.concat(df_interpolated, ignore_index=True)

missing_before = df_raw['value'].isna().sum()
missing_after = df['value'].isna().sum()

print(f"Manquants avant: {missing_before:,}")
print(f"Manquants après: {missing_after:,}")
print(f"Interpolés: {missing_before - missing_after:,}")
print(f"✅ Taux de couverture: {(1 - missing_after/len(df))*100:.1f}%")

🔄 Interpolation des valeurs manquantes...

Manquants avant: 0
Manquants après: 0
Interpolés: 0
✅ Taux de couverture: 100.0%


In [5]:
# Détection robuste d'outliers (Z-score modifié)
print("🔄 Détection d'outliers (Z-score)...\n")

df['is_outlier'] = False
outlier_count = 0

for pollutant in df['pollutant'].unique():
    if pd.notna(pollutant):
        mask = df['pollutant'] == pollutant
        values = df.loc[mask, 'value']
        
        if len(values) > 10:
            # Z-score sur valeurs non nulles
            clean_values = values.dropna()
            if len(clean_values) > 5:
                z_scores = np.abs(stats.zscore(clean_values, nan_policy='omit'))
                threshold = 3.5  # Plus strict que 3
                
                outliers = z_scores > threshold
                n_outliers = outliers.sum()
                
                if n_outliers > 0:
                    # Marquer les outliers dans le mask original
                    outlier_indices = clean_values[z_scores > threshold].index
                    df.loc[outlier_indices, 'is_outlier'] = True
                    print(f"  {pollutant}: {n_outliers} potentiels outliers détectés")
                    outlier_count += n_outliers

print(f"\n✅ Outliers marqués (non supprimés): {df['is_outlier'].sum()}")

🔄 Détection d'outliers (Z-score)...

  COV: 446 potentiels outliers détectés
  NOX: 622 potentiels outliers détectés
  PM10: 25 potentiels outliers détectés
  PM25: 19 potentiels outliers détectés
  SOX: 267 potentiels outliers détectés

✅ Outliers marqués (non supprimés): 1379


In [6]:
# Lissage exponentiel (EMA, alpha=0.3)
print("🔄 Lissage EMA (alpha=0.3)...\n")

df['value_smooth'] = df['value'].copy()

for (site, pollutant), group_indices in df.groupby(['site_id', 'pollutant']).groups.items():
    subset_indices = sorted(group_indices)
    subset = df.loc[subset_indices].sort_values('timestamp_utc')
    
    if len(subset) > 1:
        smooth = subset['value'].ewm(span=10, adjust=False).mean()
        df.loc[subset.index, 'value_smooth'] = smooth.values

print("✅ Lissage EMA appliqué")
print(f"   Avant lissage - Mean: {df['value'].mean():.2f}, Std: {df['value'].std():.2f}")
print(f"   Après lissage - Mean: {df['value_smooth'].mean():.2f}, Std: {df['value_smooth'].std():.2f}")

🔄 Lissage EMA (alpha=0.3)...

✅ Lissage EMA appliqué
   Avant lissage - Mean: 73.00, Std: 67.88
   Après lissage - Mean: 73.00, Std: 60.84


## Section 3: Feature Engineering

In [7]:
# Features statistiques — fenêtres horaires (données à pas 1 h, TRAINING_PLAN)
print("🔄 Feature engineering (fenêtres 6 h / 12 h)...\n")

for col in ['value_mean_6h', 'value_std_6h', 'value_max_12h', 'value_roc_1h',
            'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']:
    df[col] = np.nan

for (site, pollutant), group_indices in df.groupby(['site_id', 'pollutant']).groups.items():
    subset = df.loc[sorted(group_indices)].sort_values('timestamp_utc')
    if len(subset) < 2:
        continue
    s = subset['value_smooth']
    df.loc[subset.index, 'value_mean_6h'] = s.rolling(6, min_periods=1, center=True).mean().values
    df.loc[subset.index, 'value_std_6h'] = s.rolling(6, min_periods=1, center=True).std().values
    df.loc[subset.index, 'value_max_12h'] = s.rolling(12, min_periods=1, center=True).max().values
    df.loc[subset.index, 'value_roc_1h'] = s.diff().fillna(0).values
    hours = subset['timestamp_utc'].dt.hour + subset['timestamp_utc'].dt.minute / 60.0
    dow = subset['timestamp_utc'].dt.dayofweek
    df.loc[subset.index, 'hour_sin'] = np.sin(2 * np.pi * hours / 24)
    df.loc[subset.index, 'hour_cos'] = np.cos(2 * np.pi * hours / 24)
    df.loc[subset.index, 'dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df.loc[subset.index, 'dow_cos'] = np.cos(2 * np.pi * dow / 7)

print(f"✅ Features créées: {len([c for c in df.columns if c.startswith('value_')])} colonnes")
print(f"\nNouvelles colonnes:")
feature_cols = [c for c in df.columns if 'value_' in c and c != 'value']
for col in feature_cols:
    print(f"  - {col}")

🔄 Feature engineering (fenêtres 6 h / 12 h)...



✅ Features créées: 5 colonnes



Nouvelles colonnes:
  - value_smooth
  - value_mean_6h
  - value_std_6h
  - value_max_12h
  - value_roc_1h


## Section 4: Normalisation et Scaler

In [8]:
# MinMaxScaler sur value_smooth (par polluant — échelles différentes)
print("🔄 Création MinMaxScaler...\n")

df['value_normalized'] = np.nan
scaler_by_pollutant = {}

for pollutant in TARGET_POLLUTANTS:
    mask = df['pollutant'] == pollutant
    vals = df.loc[mask, 'value_smooth'].dropna()
    if len(vals) == 0:
        continue
    sc = MinMaxScaler(feature_range=(0, 1))
    scaled = sc.fit_transform(vals.values.reshape(-1, 1)).ravel()
    df.loc[vals.index, 'value_normalized'] = scaled
    scaler_by_pollutant[pollutant] = sc

joblib.dump(scaler_by_pollutant, scaler_file)
scaler = scaler_by_pollutant

# Sauvegarder scaler
joblib.dump(scaler, scaler_file)

print(f"✅ MinMaxScaler créé et sauvegardé à: {scaler_file}")
print(f"   Polluants scalés: {list(scaler_by_pollutant.keys())}")
for pol, sc in scaler_by_pollutant.items():
    print(f"  {pol}: [{sc.data_min_[0]:.2f}, {sc.data_max_[0]:.2f}]")

🔄 Création MinMaxScaler...

✅ MinMaxScaler créé et sauvegardé à: C:\Users\melik\Desktop\ia\models\scaler.pkl
   Polluants scalés: ['NOX', 'SOX', 'PM25', 'PM10', 'COV']
  NOX: [20.05, 675.39]
  SOX: [0.85, 34.05]
  PM25: [73.07, 141.00]
  PM10: [128.01, 179.92]
  COV: [1.04, 354.57]


## Section 5: Export

In [9]:
# Nettoyage final et sélection colonnes
print("🔄 Sélection colonnes finales...\n")

final_cols = [
    'timestamp_utc',
    'site_id',
    'site_name',
    'pollutant',
    'value',
    'unit',
    'value_smooth',
    'value_mean_6h',
    'value_std_6h',
    'value_max_12h',
    'value_roc_1h',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'temperature_c',
    'humidity_percent',
    'pressure_hpa',
    'wind_speed_ms',
    'source_name',
    'imputed',
    'is_outlier',
    'value_normalized'
]

# Vérifier colonnes présentes
final_cols = [c for c in final_cols if c in df.columns]

df_export = df[final_cols].copy()
df_export = df_export.dropna(subset=['value']).reset_index(drop=True)

print(f"✅ Dataset final: {len(df_export):,} lignes × {len(df_export.columns)} colonnes")
print(f"\nColonnes finales:")
for i, col in enumerate(final_cols, 1):
    print(f"  {i:2d}. {col}")

🔄 Sélection colonnes finales...

✅ Dataset final: 134,756 lignes × 23 colonnes

Colonnes finales:
   1. timestamp_utc
   2. site_id
   3. site_name
   4. pollutant
   5. value
   6. unit
   7. value_smooth
   8. value_mean_6h
   9. value_std_6h
  10. value_max_12h
  11. value_roc_1h
  12. hour_sin
  13. hour_cos
  14. dow_sin
  15. dow_cos
  16. temperature_c
  17. humidity_percent
  18. pressure_hpa
  19. wind_speed_ms
  20. source_name
  21. imputed
  22. is_outlier
  23. value_normalized


In [10]:
# Exporter
df_export.to_csv(output_file, index=False)

print(f"✅ Dataset nettoyé exporté à: {output_file}")
print(f"   Taille: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nStatistiques finales:")
print(df_export.describe())

✅ Dataset nettoyé exporté à: C:\Users\melik\Desktop\ia\data\cleaned_features.csv
   Taille: 42.55 MB

Statistiques finales:
                    timestamp_utc          value   value_smooth  \
count                      134756  134756.000000  134756.000000   
mean   2015-01-13 03:18:15.286295      72.999786      72.995814   
min           2004-03-10 18:00:00       0.000000       0.847344   
25%           2015-06-05 23:00:00      16.645700      30.152039   
50%           2016-04-15 17:00:00      53.141220      48.835492   
75%           2017-02-22 01:00:00     115.000000     104.421652   
max           2017-12-28 22:00:00     874.000000     675.385145   
std                           NaN      67.882205      60.843339   

       value_mean_6h   value_std_6h  value_max_12h   value_roc_1h  \
count  134756.000000  134756.000000  134756.000000  134756.000000   
mean       72.995566       4.469981      82.686501       0.000883   
min         1.119761       0.032633       1.810897     -67.700934

## ✅ Résumé

✅ Interpolation: valeurs manquantes interpolées (< 5%)
✅ Lissage: EMA appliqué (alpha=0.3)
✅ Features: mean_10, std_10, rate_of_change engineered
✅ Normalisation: MinMaxScaler [0,1] sauvegardé
✅ Export: cleaned_features.csv + scaler.pkl

**Prochaine étape**: Notebook 03 - Génération données synthétiques